### **Package**

In [ ]:
import subprocess
import sys
import os

print(sys.version)

for pkg in ["pymoo", "GPy", "autogluon.tabular"]:
    module_name = pkg.split('.')[0]
    try:
        __import__(pkg)
        print(f"{pkg} is already installed.")
    except ImportError:
        print(f"{pkg} is not installed. Installing...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])
        print(f"{pkg} installed.")


import warnings
warnings.filterwarnings("ignore", message=".*load_learner.*pickle.*")


In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd().resolve()
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

from src.opt_problem import build_problem
from src.survival import Survival_standard, Survival_dual_ranking
from src.data import generate_data
from src.metrics import get_metrics
from src.models import autogluon_qr_fit_predict, autogluon_qr_pred_mean_quantiles
from src.uncertainty import coverage_upper
from src.experiment import run_experiment
from src.other_functions import mean_std
from src.plotting import plot_obj_2d

from pymoo.operators.sampling.lhs import LHS
from sklearn.metrics import mean_squared_error
import numpy as np


### **Main**

###### 1. Initial settings

In [ ]:
# Problem: dtlz1-7, omnitest, bnh, truss2d, welded_beam
problem_name = 'dtlz1'
n_var = 10
n_obj = 2
problem = build_problem(
    problem_name=problem_name,
    n_var=n_var,
    n_obj=n_obj
)
print(f"Problem name: {problem_name}")
print(f"Cons: {problem.n_constr}")
print(f"Var: {n_var}")
print(f"Obj: {n_obj}")

# EC algorithm
n_gen = 100
pop_size = 100

# Data
random_seed = 42
sampling = LHS()
sample_size = 11 * n_var - 1
X_train, y_train, X_val, y_val, X_test, y_test = generate_data(
    problem=problem,
    sample_size=sample_size,
    sampling=sampling,
    train_seed=random_seed,
    val_size=100,
    test_size=100,
    test_seed=1
)
y_train_f1 = y_train[:, 0]
y_train_f2 = y_train[:, 1]
print(f"\nSampling X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_val shape: {y_val.shape}")
print(f"y_test shape: {y_test.shape}")

# Metrics
hv, igd_plus, obj_min, obj_max, ref_point = get_metrics(
    problem_name=problem_name,
    problem=problem,
    n_var=n_var,
    n_obj=n_obj
)
print('\nMin-Max normalization -> Min: ', obj_min)
print('Min-Max normalization -> Max: ', obj_max)
print('HV Reference points: ', ref_point)

np.set_printoptions(precision=4, suppress=True)


###### 2. Autogluon_QR surrogate model training

In [ ]:
qr_f1, model_qr_f1 = autogluon_qr_fit_predict(X_train, y_train_f1, X_test)
qr_f2, model_qr_f2 = autogluon_qr_fit_predict(X_train, y_train_f2, X_test)

y_q50, y_q80, y_q90, y_q95 = autogluon_qr_pred_mean_quantiles(
    model_qr_f1,
    model_qr_f2,
    X_test,
    verbose=True
)

print(f"\nAutogluon-QR MSE: {mean_squared_error(y_test, y_q50):.2e}\n")


###### 2.1 CICP

In [ ]:
print('Coverage')
for level, q_upper in [(0.90, y_q90)]:
    per_dim, overall = coverage_upper(y_test, q_upper)
    print(f"{int(level*100)}% interval: per_dim={per_dim*100}%, overall={overall*100:.1f}%")


###### 3. Optimization using QR

In [ ]:
results = run_experiment(
    problem=problem,
    problem_name=problem_name,
    n_gen=n_gen,
    pop_size=pop_size,
    model_f1=model_qr_f1,
    model_f2=model_qr_f2,
    obj_min=obj_min,
    obj_max=obj_max,
    hv=hv,
    igd_plus=igd_plus,
    use_surrogate='QR_uncertainty',
    survival_function=Survival_standard(),
    use_callback=False,
    seeds=range(1, 2)
)

mse_list = results["mse_list"]
igd_list = results["igd_list"]
hv_surrogate_list = results["hv_surrogate_list"]
hv_real_list = results["hv_real_list"]

obj = results["run_details"][-1]["obj"]
f_real = results["run_details"][-1]["f_real"]

plot_obj_2d(obj,
            xlim=(float(obj_min[0])-0.3*float(obj_max[0]), float(obj_max[0])),
            ylim=(float(obj_min[1])-0.3*float(obj_max[1]), float(obj_max[1])))
plot_obj_2d(f_real,
            xlim=(float(obj_min[0])-0.3*float(obj_max[0]), float(obj_max[0])),
            ylim=(float(obj_min[1])-0.3*float(obj_max[1]), float(obj_max[1])))


In [ ]:
# mean_mse, std_mse = mean_std(mse_list)
# mean_igd, std_igd = mean_std(igd_list)
# mean_hv_real, std_hv_real = mean_std(hv_real_list)
# mean_hv_surrogate, std_hv_surrogate = mean_std(hv_surrogate_list)

# print('Problem name: ', problem_name)
# print("\n=== Autogluon-QR ===")
# print(f"MSE: Mean = {mean_mse:.2e}, Std = {std_mse:.2e}")
# print(f"IGD+: Mean = {mean_igd:.2e}, Std = {std_igd:.2e}")
# print(f"Sur HV: Mean = {mean_hv_surrogate:.2f}, Std = {std_hv_surrogate:.2f}")
# print(f"Real HV: Mean = {mean_hv_real:.2f}, Std = {std_hv_real:.2f}")


###### 3. Optimization using QR + dual-ranking (q=0.90)

In [ ]:
results = run_experiment(
    problem=problem,
    problem_name=problem_name,
    n_gen=n_gen,
    pop_size=pop_size,
    model_f1=model_qr_f1,
    model_f2=model_qr_f2,
    obj_min=obj_min,
    obj_max=obj_max,
    hv=hv,
    igd_plus=igd_plus,
    use_surrogate='QR_uncertainty',
    survival_function=Survival_dual_ranking(alpha=0.9),
    use_callback=False,
    seeds=range(1, 2)
)

mse_list = results["mse_list"]
igd_list = results["igd_list"]
hv_surrogate_list = results["hv_surrogate_list"]
hv_real_list = results["hv_real_list"]

obj_q90 = results["run_details"][-1]["obj"]
f_real_q90 = results["run_details"][-1]["f_real"]

plot_obj_2d(obj_q90,
            xlim=(float(obj_min[0])-0.3*float(obj_max[0]), float(obj_max[0])),
            ylim=(float(obj_min[1])-0.3*float(obj_max[1]), float(obj_max[1])))
plot_obj_2d(f_real_q90,
            xlim=(float(obj_min[0])-0.3*float(obj_max[0]), float(obj_max[0])),
            ylim=(float(obj_min[1])-0.3*float(obj_max[1]), float(obj_max[1])))


In [ ]:
# mean_mse, std_mse = mean_std(mse_list)
# mean_igd, std_igd = mean_std(igd_list)
# mean_hv_real, std_hv_real = mean_std(hv_real_list)
# mean_hv_surrogate, std_hv_surrogate = mean_std(hv_surrogate_list)

# print('Problem name: ', problem_name)
# print("\n=== Autogluon-QR + dual-ranking (q=0.90) ===")
# print(f"MSE: Mean = {mean_mse:.2e}, Std = {std_mse:.2e}")
# print(f"IGD+: Mean = {mean_igd:.2e}, Std = {std_igd:.2e}")
# print(f"Sur HV: Mean = {mean_hv_surrogate:.2f}, Std = {std_hv_surrogate:.2f}")
# print(f"Real HV: Mean = {mean_hv_real:.2f}, Std = {std_hv_real:.2f}")
